# Yield Curve Dynamics — Colab Training

Run cells top-to-bottom.

**Runtime:** `Runtime > Change runtime type > GPU` (recommended for Stage B constraints)

**Note:** Raw/processed data is **not** in GitHub (gitignored). This notebook downloads FRED data and preprocesses it automatically on first run.

**Plots:** saved under `reports/figures/` and `reports/comparison/figures/`, and shown inline in cells **5**, **9**, and **10**. After Drive copy (cell 8): `My Drive/yield-curve-geometric-sde/reports/`.

In [ ]:
# Clone repo (skip if you already uploaded the project)
import os

REPO_URL = "https://github.com/danield116/Yield-Curve-Dynamics-ML.git"
REPO_DIR = "/content/Yield-Curve-Dynamics-ML"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

PROJECT_DIR = f"{REPO_DIR}/yield-curve-geometric-sde"
%cd {PROJECT_DIR}
print("Working directory:", os.getcwd())

In [ ]:
# Install Python dependencies (torch is usually preinstalled on Colab)
!pip install -q pyyaml pandas numpy scipy scikit-learn matplotlib seaborn tqdm statsmodels

In [ ]:
# Optional: mount Google Drive to persist checkpoints across sessions
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/yield-curve-geometric-sde"
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("Drive root:", DRIVE_ROOT)

In [ ]:
# Download FRED yields + preprocess (creates data/raw and data/processed)
import os
from pathlib import Path

# Colab resets cwd on reconnect — always jump back to project root.
PROJECT_DIR = "/content/Yield-Curve-Dynamics-ML/yield-curve-geometric-sde"
os.chdir(PROJECT_DIR)
print("Working directory:", os.getcwd())

!python data/download_fred_yields.py \
  --start-date 2001-07-01 \
  --output-path data/raw/fred_yields.csv

!python data/preprocess_curves.py \
  --input-path data/raw/fred_yields.csv \
  --output-dir data/processed \
  --levelscript

processed = Path("data/processed")
expected = ["train_scaled.csv", "val_scaled.csv", "test_scaled.csv"]
missing = [name for name in expected if not (processed / name).exists()]
if missing:
    raise FileNotFoundError(
        f"Preprocess did not create: {missing}. "
        "Scroll up in this cell for download/preprocess errors, then re-run."
    )
print("Processed files OK:", [f.name for f in sorted(processed.glob("*.csv"))])

In [ ]:
# Quick sanity check: data shapes + split plot
import os
import pandas as pd
from pathlib import Path
from IPython.display import Image, display

PROJECT_DIR = "/content/Yield-Curve-Dynamics-ML/yield-curve-geometric-sde"
os.chdir(PROJECT_DIR)

processed = Path("data/processed")
required = ["train_scaled.csv", "val_scaled.csv", "test_scaled.csv"]
missing = [name for name in required if not (processed / name).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} under {processed.resolve()}. "
        "Run the previous cell (FRED download + preprocess) first and fix any errors there."
    )

train = pd.read_csv(processed / "train_scaled.csv", index_col=0, parse_dates=True)
val = pd.read_csv(processed / "val_scaled.csv", index_col=0, parse_dates=True)
test = pd.read_csv(processed / "test_scaled.csv", index_col=0, parse_dates=True)

print(f"train: {train.shape} | val: {val.shape} | test: {test.shape}")
print(f"train dates: {train.index.min().date()} -> {train.index.max().date()}")

split_png = Path("reports/figures/split_boundaries_scaled.png")
!python data/visualize_splits.py \
  --processed-dir data/processed \
  --suffix scaled \
  --output-path {split_png}

print(f"Saved: {split_png.resolve()}")
display(Image(filename=str(split_png), width=800))

In [ ]:
# Stage A: train manifold model (Student-t CVAE by default)
import torch
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

!python training/train_stage_a.py --config config/default.yaml

In [ ]:
# Stage B ablations
# NOTE: latent_dim/hidden_dim changed (5 / 256) -> ALL old checkpoints are invalid.
#       You MUST rerun Stage A (cell above) before ANY mode here.
# TRAIN_MODE:
#   "fast"                 -> sde_only only (~1-2h GPU)
#   "jacobian_focus"       -> retrain sde_jacobian only with stronger Jacobian (~1-2h)
#   "jacobian_vs_only"     -> sde_only + sde_jacobian head-to-head (~2-3h)
#   "sde_both_retrain"     -> tuned sde_both only (~1-2h)
#   "constraint_ablations" -> retrain sde_pde, sde_jacobian, sde_both with tuned weights (~3-4h)
#   "full"                 -> all 4 ablations (~6+h GPU) -- use this after a fresh Stage A
TRAIN_MODE = "full"

from pathlib import Path

if TRAIN_MODE == "fast":
    ablations = ["sde_only"]
elif TRAIN_MODE == "jacobian_focus":
    ablations = ["sde_jacobian"]
elif TRAIN_MODE == "jacobian_vs_only":
    ablations = ["sde_only", "sde_jacobian"]
elif TRAIN_MODE == "sde_both_retrain":
    ablations = ["sde_both"]
elif TRAIN_MODE == "constraint_ablations":
    ablations = ["sde_pde", "sde_jacobian", "sde_both"]
else:
    ablations = ["sde_only", "sde_pde", "sde_jacobian", "sde_both"]

print("TRAIN_MODE:", TRAIN_MODE, "| ablations:", ablations)

for ablation in ablations:
    print("\n" + "=" * 60)
    print(f"Training Stage B ablation: {ablation}")
    print("=" * 60)
    !python training/train_stage_b.py --config config/default.yaml --ablation {ablation}

ckpts = sorted(Path("reports/checkpoints/stage_b").glob("stage_b_*_best.pt"))
print(f"\nStage B checkpoints saved: {len(ckpts)}")
for p in ckpts:
    print(" ", p)
if not ckpts:
    raise FileNotFoundError("No Stage B checkpoints found. Scroll up for training errors.")

In [ ]:
# Optional: copy artifacts to Google Drive
import shutil

for folder in [
    "reports/checkpoints",
    "reports/latents",
    "reports/forecasts",
    "reports/figures",
    "reports/comparison",
]:
    src = Path(folder)
    if src.exists():
        dst = Path(DRIVE_ROOT) / folder
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"Copied {src} -> {dst}")

print("Done. Artifacts saved to Drive.")
print("Plots on Drive:", list((Path(DRIVE_ROOT) / "reports/comparison/figures").glob("*.png")))

In [ ]:
# Evaluate all models + baselines, save scorecard, and SHOW comparison plots inline
import os
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

PROJECT_DIR = "/content/Yield-Curve-Dynamics-ML/yield-curve-geometric-sde"
os.chdir(PROJECT_DIR)

stage_b_ckpts = sorted(Path("reports/checkpoints/stage_b").glob("stage_b_*_best.pt"))
print(f"Stage B checkpoints found: {len(stage_b_ckpts)}")
for p in stage_b_ckpts:
    print(" ", p.name)
if not stage_b_ckpts:
    raise FileNotFoundError(
        "No Stage B checkpoints. Run cell 7 to completion first (set FAST_TRAIN=True for a quicker sde_only run)."
    )

!python evaluation/evaluate_run.py --config config/default.yaml --split test --output-dir reports/comparison

scorecard = pd.read_csv("reports/comparison/scorecard.csv")
print("=== Scorecard (lower curve_rmse = better) ===")
display(scorecard[["model", "horizon", "curve_rmse", "curve_mae_mean"]].sort_values(["horizon", "curve_rmse"]))

for h in [1, 5, 21]:
    constraint_path = Path(f"reports/comparison/constraint_ablation_h{h}.csv")
    if constraint_path.exists():
        ablation_h = pd.read_csv(constraint_path)
        print(f"\n=== Constraint ablation @ h={h} (Jacobian vs PDE geometry) ===")
        display(ablation_h)
    else:
        print(f"No constraint_ablation_h{h}.csv — re-run evaluate_run after git pull.")

figures_dir = Path("reports/comparison/figures")
pngs = sorted(figures_dir.glob("*.png")) if figures_dir.exists() else []
print(f"\nSaved {len(pngs)} plots to: {figures_dir.resolve()}")
for png in pngs:
    print(f"  - {png.name}")
    display(Image(filename=str(png), width=800))

In [ ]:
# View plots only (no retrain) — run this anytime after cells 6-7 complete
# Builds training-curve PNGs from history JSON if comparison plots are missing.
import os
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import Image, display

from visualization.plot_curves import plot_training_history

PROJECT_DIR = "/content/Yield-Curve-Dynamics-ML/yield-curve-geometric-sde"
os.chdir(PROJECT_DIR)

figures_dir = Path("reports/comparison/figures")
figures_dir.mkdir(parents=True, exist_ok=True)

# Training loss curves from checkpoint history JSON
stage_a_hist = Path("reports/checkpoints/stage_a/stage_a_student_t_cvae_history.json")
if stage_a_hist.exists():
    plot_training_history(stage_a_hist, output_path=figures_dir / "stage_a_val_loss.png")
    plt.show()

for hist in sorted(Path("reports/checkpoints/stage_b").glob("stage_b_*_history.json")):
    plot_training_history(hist, output_path=figures_dir / f"{hist.stem}.png")
    plt.show()

# Display every saved PNG
for folder in [Path("reports/figures"), figures_dir]:
    if not folder.exists():
        continue
    pngs = sorted(folder.glob("*.png"))
    if not pngs:
        continue
    print(f"\n=== {folder} ===")
    for png in pngs:
        print(png.name)
        display(Image(filename=str(png), width=800))

if not list(figures_dir.glob("*.png")):
    print("No comparison plots yet. Run cell 9 once (evaluate_run) to generate model comparison charts.")